# Run AtlasTailor end to end on 3D zebrafish weMERFISH data

This tutorial runs one real reciprocal reconstruction direction at 50%
epiboly: embryo E1 is the deeply measured reference and embryo E2 is the
target. Both H5AD files are public on Dryad. AtlasTailor selects exactly 16
genes from E1, registers geometry without target expression, freezes the E2
prediction and opens E2 non-anchor expression only for evaluation.

The notebook executes the reusable AtlasTailor API. The six-direction frozen
manuscript analysis and its original prediction locks remain in the companion
reproducibility repository.

## 1. Download and verify the public data

Download these two files from
[Dryad 10.5061/dryad.j0zpc86v9](https://doi.org/10.5061/dryad.j0zpc86v9)
and place them in `data/external/wemerfish/`:

- `weMERFISH_measured_A_50p_E1.h5ad`
- `weMERFISH_measured_A_50p_E2.h5ad`

Dryad may require an interactive public-download session for large files. This
notebook does not automate browser verification. The next cell verifies the
exact frozen SHA-256 hashes before any analysis.

In [ ]:
from __future__ import annotations

import hashlib
import os
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import hyperspatial as hs
from hyperspatial.io import read_reference, read_target, read_truth
from hyperspatial.metrics import evaluate_matrices

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = Path.cwd().parent


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def new_output(label: str) -> Path:
    root = Path(os.environ.get("ATLASTAILOR_RUNS", ROOT / "tutorial_runs"))
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    output = root / f"{label}_{stamp}"
    output.mkdir(parents=True, exist_ok=False)
    return output

DATA = Path(os.environ.get("ATLASTAILOR_WEMERFISH_DATA", ROOT / "data" / "external" / "wemerfish"))
SOURCE = DATA / "weMERFISH_measured_A_50p_E1.h5ad"
TARGET = DATA / "weMERFISH_measured_A_50p_E2.h5ad"
EXPECTED = {
    SOURCE.name: "8c579f6d1278ec12e17665b6645c892890b29d8e2500cb4a3eecafcafc620884",
    TARGET.name: "e0a88d080cc5c5750c94ff7e2dfb313f257f1a4beaeae03c76d9a466f7f7c2d6",
}

for path in (SOURCE, TARGET):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Download it from Dryad DOI 10.5061/dryad.j0zpc86v9."
        )
    observed = sha256(path)
    if observed != EXPECTED[path.name]:
        raise RuntimeError(f"SHA-256 mismatch for {path.name}: {observed}")

pd.DataFrame([{"file": path.name, "bytes": path.stat().st_size, "sha256": sha256(path)}
              for path in (SOURCE, TARGET)])

## 2. Source-only 16-gene panel design

Only E1 is passed to the panel-design function. The 3D coordinate field is
read from `obsm['global_sphere']`, matching the public deposit.

In [ ]:
run_root = new_output("wemerfish_50p_E1_to_E2")
reference = read_reference(SOURCE, coordinate_key="global_sphere")
design = hs.design_panel(
    reference,
    config=hs.DesignConfig(budget=16, budgets=(8, 16, 32)),
    out=run_root / "panel_design",
)
assert len(design.genes) == 16
design.ordered_panel[["position", "gene"]]

## 3. Anchor-only target loading, adaptation and prediction lock

Although the public E2 file contains its full transcriptome, `read_target`
opens it in backed mode and materializes only the declared 16 columns. E2
non-anchor expression is therefore unavailable to registration, fitting,
operator selection, calibration and guard decisions.

In [ ]:
target_anchors = read_target(TARGET, design.genes, coordinate_key="global_sphere")
assert target_anchors.measured_expression.shape[1] == 16
assert target_anchors.metadata["target_nonanchor_materialized"] is False

result = hs.adapt(
    reference,
    target_anchors,
    design.genes,
    out=run_root / "adaptation",
    config=hs.AdaptConfig(registration="geometry", seeds=(42, 43, 44)),
)
assert result.manifest.status == "completed"
assert result.manifest.data["target_nonanchor_access_before_prediction"] is False
{
    "run": str(result.manifest.run_dir),
    "predicted_genes": len(result.genes),
    "source_supported_corrections": int(result.support.supported.sum()),
    "manifest_status": result.manifest.status,
}

## 4. Post-lock evaluation

The target truth is opened only after the completed prediction manifest has
been asserted. This evaluation is a fresh software tutorial run; manuscript
numbers should be taken from the frozen results walkthrough and companion
reproducibility repository.

In [ ]:
truth, truth_coordinates = read_truth(TARGET, result.genes, coordinate_key="global_sphere")
np.testing.assert_allclose(truth_coordinates, result.coordinates)
map_metrics = hs.validate(result, truth, out=run_root / "map_metrics.tsv")
idw_metrics = evaluate_matrices(
    truth, result.registered_atlas, result.registered_atlas, result.coordinates
)
metrics = pd.DataFrame({
    "gene": result.genes,
    "IDW_spatial_Pearson": idw_metrics.spatial_pearson,
    "MAP_spatial_Pearson": map_metrics.spatial_pearson,
    "IDW_log1p_RMSE": idw_metrics.log1p_rmse,
    "MAP_log1p_RMSE": map_metrics.log1p_rmse,
    "MAP_residual_Pearson": map_metrics.residual_pearson,
})
metrics.to_csv(run_root / "post_lock_comparison.tsv", sep="\t", index=False)
hidden = ~metrics.gene.isin(design.genes)
summary = pd.Series({
    "hidden_genes": int(hidden.sum()),
    "IDW_median_spatial_Pearson": metrics.loc[hidden, "IDW_spatial_Pearson"].median(),
    "MAP_median_spatial_Pearson": metrics.loc[hidden, "MAP_spatial_Pearson"].median(),
    "IDW_log1p_RMSE": np.sqrt(np.mean((truth[:, hidden] - result.registered_atlas[:, hidden]) ** 2)),
    "MAP_log1p_RMSE": np.sqrt(np.mean((truth[:, hidden] - result.adapted[:, hidden]) ** 2)),
})
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
axes[0].scatter(metrics.loc[hidden, "IDW_spatial_Pearson"],
                metrics.loc[hidden, "MAP_spatial_Pearson"], s=10, alpha=0.5)
limits = [np.nanmin(metrics.loc[hidden, ["IDW_spatial_Pearson", "MAP_spatial_Pearson"]].to_numpy()),
          np.nanmax(metrics.loc[hidden, ["IDW_spatial_Pearson", "MAP_spatial_Pearson"]].to_numpy())]
axes[0].plot(limits, limits, color="0.35", lw=1)
axes[0].set(xlabel="Registered IDW spatial Pearson", ylabel="AtlasTailor spatial Pearson")
axes[1].scatter(metrics.loc[hidden, "IDW_log1p_RMSE"], metrics.loc[hidden, "MAP_log1p_RMSE"],
                s=10, alpha=0.5)
limits = [np.nanmin(metrics.loc[hidden, ["IDW_log1p_RMSE", "MAP_log1p_RMSE"]].to_numpy()),
          np.nanmax(metrics.loc[hidden, ["IDW_log1p_RMSE", "MAP_log1p_RMSE"]].to_numpy())]
axes[1].plot(limits, limits, color="0.35", lw=1)
axes[1].set(xlabel="Registered IDW log1p RMSE", ylabel="AtlasTailor log1p RMSE")
fig.tight_layout()